
# Dimensionality reduction and clustering of Cell Painting profiles

**Dataset.** EU-OPENSCREEN bioactives, `IMTM_HepG2` subset ([Wolff et al.
2024](https://doi.org/10.1101/2024.08.27.609964)) — 2,459 bioactive compounds imaged in
HepG2 cells at the IMTM site, built into an AnnData by
[`scverse/cell-painting-io`](https://github.com/scverse/cell-painting-io/blob/cpjump1-anndata-notebook/eu_os_bioactives_to_anndata.ipynb).
**10,668 wells x 2,776 features**: 7 library plates (`B1001`-`B1007`) x 4 replicate
plates (`R1`-`R4`) x 384 wells.

**The substitution that makes scverse work here.** A *well* is an observation and a
*CellProfiler feature* is a variable. Everything downstream of PCA — neighbour graphs,
UMAP, Leiden, marker detection — is agnostic to what the columns mean, so it transfers
unchanged. What does *not* transfer is everything upstream: counts, library size,
log-transforms, and the mean-variance relationship that highly-variable-gene selection
is built on.

This notebook walks the standard
[single-cell best practices](https://www.sc-best-practices.org/) dimensionality-reduction
and clustering path, and at each step states the assumption being made, whether it
survives the move to morphological profiles, and what to do when it doesn't.

**Backbone notebook.** Cells are meant to be run, inspected and edited. Places where
you should make a judgement call are marked **TODO**.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1
sc.set_figure_params(dpi=90, frameon=False, figsize=(4, 4))

# Override with EU_OS_H5AD if your copy lives elsewhere.
H5AD = Path(os.environ.get("EU_OS_H5AD", "../data/eu_os_imtm_hepg2.h5ad"))
CLIP = 10.0  # see the heavy-tail section of notebook 01


## 2. Load and get oriented

Read the object and look at what is already there before touching it.

In [ ]:

adata = ad.read_h5ad(H5AD)
adata

In [ ]:

# obs: one row per well. Plate/Replicate/Well locate it, EOS identifies the compound.
adata.obs.drop(columns=["smiles", "target_genes"]).head()

In [ ]:

# var: the feature axis is annotated, which is what makes notebook 02 possible.
print(adata.var.head(3), "\n")
print(pd.crosstab(adata.var.family, adata.var.compartment))
print("\nchannels:", adata.var.channel.value_counts(dropna=False).to_dict())


Two things to notice, both consequential.

**The stain has four channels, not five.** `DNA`, `ER`, `AGP`, `Mito` — there is no RNA
channel in this study. If you port code from a JUMP or CPJUMP1 notebook that iterates
over five channels, it will silently produce empty groups.

**The feature axis is wildly imbalanced.** `Texture` alone is 1,872 of 2,776 features
(67%), because texture is computed per channel x scale x angle. `ObjectSkeleton` is a
single feature. Any analysis that treats features as exchangeable — PCA, correlation
pruning, enrichment — is dominated by texture unless you intervene.


## 3. What the matrix already contains

> **Assumption (single-cell).** You start from raw integer counts and run QC ->
> size-factor normalization -> `log1p` -> HVG -> scale -> PCA.
>
> **Cell Painting.** The first three are already done, and done differently. `.X` here is
> a **per-plate robust z-score against DMSO**: for each replicate plate, subtract the
> DMSO median and divide by the DMSO MAD, feature by feature
> (`pycytominer`'s `mad_robustize`). The raw per-well medians are kept in
> `layers["aggregated"]`.
>
> **Consequence.** Re-running the transcriptomics preamble is not just unnecessary, it
> is wrong. `log1p` is undefined on negative values. There is no library size to
> normalize by — the control wells, not the total signal, are the reference. And
> `sc.pp.scale` would replace the per-plate DMSO reference with a global one, discarding
> the batch correction that is already baked in.

In [ ]:

X, raw = adata.X, adata.layers["aggregated"]
print(f".X            mean={X.mean():+.3f}  std={X.std():.2f}  min={X.min():.0f}  max={X.max():.0f}")
print(f"layers[aggr]  mean={raw.mean():+.1f}  std={raw.std():.1f}  (raw per-well medians, native units)")
print(f"non-finite in .X: {int((~np.isfinite(X)).sum())}")


| standard step | do it here? | why |
| --- | --- | --- |
| ambient / doublet removal | no | no droplets; the analogous artifacts are out-of-focus fields and mis-segmentation, handled upstream |
| size-factor normalization | **no** | no library size; DMSO controls are the reference |
| `log1p` | **no** | values are signed z-scores |
| HVG selection | replace | no mean-variance trend to fit — see section 5 |
| `sc.pp.scale` | **no** | already unit-MAD per plate; would destroy the per-plate reference |
| PCA / neighbours / UMAP / Leiden | **yes** | unchanged |


## 4. Heavy tails: the one Cell-Painting-specific fix you cannot skip

`mad_robustize` divides by the MAD of the DMSO wells *on that plate*. For a feature that
barely moves in DMSO, that denominator is tiny, and the quotient explodes. The result is
a matrix whose extreme values are three orders of magnitude beyond its bulk.

In [ ]:

q = np.percentile(adata.X, [0.1, 1, 50, 99, 99.9])
print(f"percentiles 0.1/1/50/99/99.9: {np.round(q, 2)}")
print(f"min {adata.X.min():.0f}   max {adata.X.max():.0f}")
print(f"share |x| > 10:  {(np.abs(adata.X) > 10).mean():.3%}")
print(f"share |x| > 100: {(np.abs(adata.X) > 100).mean():.4%}")


Fewer than 2% of values exceed 10 in absolute value, and 0.01% exceed 100 — but PCA
minimises squared error, so those few values decide the components. Run PCA both ways
and compare.

In [ ]:

from sklearn.decomposition import PCA


def pca_probe(matrix, label):
    p = PCA(n_components=20, svd_solver="randomized", random_state=0).fit(matrix)
    ev = p.explained_variance_ratio_
    top = np.argsort(np.abs(p.components_[0]))[::-1][:4]
    print(f"[{label:>12}] PC1={ev[0]:6.1%}  PC2={ev[1]:5.1%}  PC1-20={ev.sum():5.1%}")
    print(f"{'':15}PC1 driven by: {', '.join(adata.var_names[top])}")


pca_probe(adata.X.astype(np.float64), "unclipped")
pca_probe(np.clip(adata.X, -CLIP, CLIP).astype(np.float64), f"clip +/-{CLIP:.0f}")


Unclipped, **PC1 absorbs ~93% of the variance** and is driven by `ObjectSkeleton` and
`Neighbors` features — the ones whose DMSO MAD is near zero. That is not a phenotype, it
is a division artifact. After clipping, PC1 falls to ~31% and its top loadings are
`Texture` features on DNA and Mito, which is a plausible morphological axis.

> **What to do.** Clipping is the blunt fix and is what we use below. Alternatives, in
> rough order of how principled they are:
>
> - **Drop unstable features**: compute each feature's DMSO MAD per plate and remove
>   features whose MAD is in the bottom few percent on any plate. Attacks the cause
>   rather than the symptom. (The upstream notebook already dropped 174 features whose
>   MAD was exactly zero; near-zero ones remain.)
> - **Rank / quantile transform** each feature (`sklearn.preprocessing.QuantileTransformer`)
>   — fully outlier-immune, but discards effect magnitude, which you need for hit calling.
> - **Winsorize per feature** at its own 1st/99th percentile instead of a global cut.
> - **Robust PCA** or a Huber loss, if you want to keep the tails and still get stable
>   components.
>
> **TODO:** try one of these and see whether the confound audit in section 8 improves.

In [ ]:

adata.layers["robust_z"] = adata.X.copy()  # keep the unclipped values
adata.X = np.clip(adata.X, -CLIP, CLIP)
print(f"clipped to +/-{CLIP:.0f}; std now {adata.X.std():.2f}")


## 5. Feature selection: the HVG analogue

> **Assumption (single-cell).** HVG selection fits a mean-variance relationship — in
> counts, variance grows with mean — and keeps genes that are more variable than that
> trend predicts. It assumes most genes are uninformative noise.
>
> **Cell Painting.** There is no mean-variance trend to fit: every feature is already
> centred and scaled to the DMSO distribution, so all means are ~0 and all DMSO
> variances are ~1 by construction. A feature's variance across *all* wells therefore
> measures something different and more directly useful — how far compounds push it
> relative to control noise.
>
> **What replaces it.** Nothing, or `pycytominer.feature_select`. The field's standard is
> to prune *redundancy* rather than select *variability*, because texture features are
> near-duplicates of each other across scales and angles. `sc.pp.highly_variable_genes`
> will run on this matrix and return something, but its `seurat`/`cell_ranger` flavours
> bin by mean expression and are meaningless here.

In [ ]:

# Variance across all wells: informative here, but note that the top of this list is
# also where unstable features hide. Compare against each feature's DMSO variance.
is_dmso = (adata.obs.pert_type == "negcon").to_numpy()
var_all = adata.X.var(axis=0)
var_dmso = adata.X[is_dmso].var(axis=0)
ratio = var_all / np.maximum(var_dmso, 1e-6)

sel = pd.DataFrame(
    {"family": adata.var.family.to_numpy(), "var_all": var_all, "var_dmso": var_dmso, "ratio": ratio},
    index=adata.var_names,
)
print(sel.sort_values("ratio", ascending=False).head(8).round(2), "\n")
print("median variance ratio by family:")
print(sel.groupby("family", observed=True).ratio.median().sort_values(ascending=False).round(2))


A high `ratio` means a feature moves much more across the library than it does among
DMSO wells — the closest honest analogue of "highly variable".

> **TODO.** Decide whether to subset. Options: keep everything (PCA will handle the
> redundancy, at the cost of interpretability); keep the top *n* by `ratio`; or run
> `pycytominer.feature_select` with `correlation_threshold` to drop near-duplicate
> texture features. Redundancy matters most for notebook 02, where correlated features
> inside a family inflate enrichment scores.


## 6. PCA

> **Assumption.** PCA wants roughly comparable feature scales and approximately
> symmetric distributions. Standard practice scales genes to unit variance first.
>
> **Cell Painting.** Already satisfied *by plate*, so pass `zero_center=True` but do not
> re-scale. The remaining violation is redundancy: with 1,872 texture features, the
> leading PCs describe texture covariance more than they describe biology.

In [ ]:

sc.pp.pca(adata, n_comps=50, zero_center=True, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
print("cumulative variance:", np.cumsum(adata.uns["pca"]["variance_ratio"])[[9, 19, 49]].round(3))


## 7. Neighbour graph and UMAP

These steps are entirely agnostic to what the features mean, so they transfer without
modification. The only Cell-Painting-specific choice is the metric: profiles are dense,
signed and continuous, so Euclidean distance in PC space is appropriate (unlike
correlation-based metrics often used on raw profiles).

In [ ]:

sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)
sc.tl.umap(adata)


## 8. Confound audit — before you look at the embedding

This is the step with no real single-cell counterpart, and the one most worth your time.
In transcriptomics, batch is a nuisance you correct and move on. In a Cell Painting
screen there are three nuisances that look exactly like biology:

1. **Plate** — reagent age, incubation, imaging session.
2. **Well position** — edge evaporation, dispensing gradients. The 384 well positions
   are shared across every plate, so a position effect is *systematic*, not random.
3. **Cell count** — confluence and cytotoxicity change morphology globally. A toxic
   compound and a slow-growing one look alike.

Quantify each before interpreting anything.

In [ ]:

def eta_squared(adata, covariate, n_pcs=20):
    """Fraction of each PC's variance explained by a categorical covariate."""
    pcs = adata.obsm["X_pca"][:, :n_pcs]
    g = pd.Series(adata.obs[covariate].astype(str).to_numpy(), name="g")
    out = []
    for j in range(n_pcs):
        y = pcs[:, j]
        grand = y.mean()
        stats = pd.DataFrame({"y": y, "g": g}).groupby("g", observed=True)["y"].agg(["mean", "size"])
        ssb = (stats["size"] * (stats["mean"] - grand) ** 2).sum()
        out.append(ssb / ((y - grand) ** 2).sum())
    return np.asarray(out)


rows = {}
for cov in ["Plate", "Replicate", "Well", "pert_type"]:
    e = eta_squared(adata, cov)
    rows[cov] = {"PC1": e[0], "PC2": e[1], "mean_PC1_20": e.mean()}
audit = pd.DataFrame(rows).T.round(3)

r_count = np.corrcoef(adata.obsm["X_pca"][:, 0], adata.obs.cell_count)[0, 1]
print(audit, "\n")
print(f"corr(PC1, cell_count) = {r_count:+.3f}")


Read this table carefully — the ordering is the finding.

**Well position explains more variance than plate does** (mean eta^2 across PC1-20 of
roughly 0.16 versus 0.03). Per-plate normalization removed the plate offset but cannot
remove a gradient that recurs at the same coordinates on every plate.

**`cell_count` correlates with PC1 at about r = 0.57.** The single largest axis of
variation in this screen is substantially confluence and toxicity. Any cluster you find
must be checked against cell count before you call it a mechanism.

> **What to do.**
>
> - **Regress out cell count** (`sc.pp.regress_out(adata, ["cell_count"])`) if you want
>   mechanism-specific signal. It also removes genuine antiproliferative biology, so keep
>   both versions and compare.
> - **Keep it as a covariate** and report it alongside every cluster, rather than
>   removing it. Usually the more honest option.
> - **Treat position as spatial.** Well row/column are real 2D coordinates, so
>   `squidpy.gr.spatial_autocorr` on a plate graph tests for edge effects directly. Out
>   of scope here, but this is the right tool.
> - **Do not** reach for Harmony or scVI on `Plate` reflexively — the plate offset is
>   already gone, and what remains is positional.
>
> **TODO:** pick one, rerun sections 6-7, and see how the audit changes.

In [ ]:

adata.obs["log_cell_count"] = np.log10(adata.obs.cell_count.clip(lower=1))
sc.pl.umap(adata, color=["Plate", "Replicate", "pert_type", "log_cell_count"], ncols=2, size=6)


## 9. Clustering

> **Assumption (single-cell).** Clusters approximate discrete cell types. The population
> is a mixture of a modest number of well-separated states, and nearly every cell
> belongs to one.
>
> **Cell Painting.** Badly violated, in a specific and predictable way. Perturbation
> response is *continuous* (a dose-dependent push away from control) and *mostly absent*
> — at a single 10 uM dose, most bioactives do nothing measurable. So expect one enormous
> "indistinguishable from DMSO" cluster plus a handful of small clusters of genuinely
> active compounds, not a partition into balanced types.
>
> **How to read a cluster.** Not as a cell type but as a **shared morphological state**.
> Chemically unrelated compounds landing in one cluster is the interesting signal — it is
> the basis of guilt-by-association MOA prediction. Wells of the *same* compound landing
> together is a reproducibility check, not a discovery.

In [ ]:

for res in (0.25, 0.5, 1.0):
    key = f"leiden_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key, flavor="igraph", n_iterations=2)
    sizes = adata.obs[key].value_counts()
    print(f"res={res}: {len(sizes)} clusters | largest {sizes.iloc[0]:>5} wells "
          f"({sizes.iloc[0] / adata.n_obs:.0%}) | singleton-ish (<20): {(sizes < 20).sum()}")

In [ ]:

LEIDEN = "leiden_0.5"  # TODO: pick a resolution once you have seen the sizes above
sc.pl.umap(adata, color=[LEIDEN], legend_loc="on data", size=6)


## 10. Is the structure real? Neighbour enrichment

A UMAP of mostly-inactive compounds is very easy to over-read. Before interpreting it,
measure whether neighbouring wells actually share labels more often than chance. This
diagnostic comes from the upstream `cell-painting-io` notebook and is worth keeping in
every Cell Painting analysis.

In [ ]:

def neighbour_enrichment(adata, keys):
    """Observed vs chance rate of neighbouring wells sharing a label."""
    graph = adata.obsp["connectivities"].tocoo()
    rows = []
    for key in keys:
        labels = adata.obs[key].to_numpy()
        observed = float((labels[graph.row] == labels[graph.col]).mean())
        baseline = float((adata.obs[key].value_counts(normalize=True).to_numpy() ** 2).sum())
        rows.append({"covariate": key, "observed": observed, "chance": baseline, "ratio": observed / baseline})
    return pd.DataFrame(rows).set_index("covariate").round(3)


neighbour_enrichment(adata, ["Plate", "Replicate", "Well", "pert_type", "EOS"])


Same-compound (`EOS`) wells are strongly enriched — around 15x chance — and `Plate` around
2.7x, so both biology and batch are visible in the graph. But `Well` comes out highest of
all at roughly 22x, which deserves care rather than alarm, because **the ratio is sensitive
to how many categories a label has.** `Well` has 384 values and a chance baseline of 0.003;
`EOS` has 2,459 values and a baseline of 0.006. Comparing ratios across labels with such
different cardinality is not apples to apples.

Look at the absolute rates instead, where the ordering flips: neighbouring wells share a
compound **8.5%** of the time and a well position **5.8%**. So compound identity is the
stronger effect, and well position is a real, substantial, but secondary one. Notebook 03
measures the same thing far more directly, and agrees: replicate wells of a compound have
mean similarity +0.61, while wells merely sharing a plate coordinate reach only +0.12.

Both numbers are small in absolute terms, and that is the honest headline. Most compounds
at a single 10 uM dose move morphology too little to pull apart at all — which is why
screens score compounds by replicate reproducibility and induction (notebook 03) rather
than by clustering.

> **What to take from this.** Do not accept "these compounds cluster, so they share a
> mechanism" without checking position and cell count. Rerun this diagnostic after
> regressing out `cell_count`, and watch which ratios move — that tells you which of your
> structure was confound.


## 11. Characterizing clusters

> **Assumption.** `sc.tl.rank_genes_groups` defaults to a t-test on log-normalised
> counts; the `wilcoxon` and `logreg` options make fewer distributional assumptions.
> "Log fold change" presumes positive values on a multiplicative scale.
>
> **Cell Painting.** Use `wilcoxon` — it only needs ranks, which is safe for signed
> z-scores with heavy tails. **Ignore `logfoldchanges`**: it is computed as a ratio of
> means that can straddle zero, so it is meaningless here. Use `scores` (the rank
> statistic) or a plain difference in medians instead.

In [ ]:

sc.tl.rank_genes_groups(adata, groupby=LEIDEN, method="wilcoxon", n_genes=15)

top = sc.get.rank_genes_groups_df(adata, group=None)
top = top.merge(adata.var[["compartment", "family", "channel"]], left_on="names", right_index=True)
print(top.groupby("group", observed=True).head(3)[
    ["group", "names", "scores", "pvals_adj", "family", "channel"]
].to_string(index=False))

In [ ]:

# Which feature families define each cluster? Far more legible than feature names.
fam = (
    top.groupby(["group", "family"], observed=True).size()
    .unstack(fill_value=0)
    .pipe(lambda d: d.div(d.sum(axis=1), axis=0))
)
print("fraction of each cluster's top features by family:")
print(fam.round(2).to_string())


> **Suggestion.** Counting family membership among top features is a crude summary. The
> principled version is to score the families directly, per well, which is what
> notebook 02 does with `decoupler`.


## 12. What are the clusters made of?

Three questions worth asking of any Cell Painting clustering, in increasing order of
how much they tell you.

In [ ]:

obs = adata.obs
print("1. Is a cluster just a confound? cell count and plate composition per cluster")
print(obs.groupby(LEIDEN, observed=True).agg(
    n=("EOS", "size"), median_cells=("cell_count", "median"),
    n_plates=("Plate", "nunique"), frac_dmso=("pert_type", lambda s: (s == "negcon").mean()),
).round(2).to_string())

In [ ]:

print("2. Do the 4 replicates of a compound co-cluster? (reproducibility)")
trt = obs[obs.pert_type == "trt"]
per_cmp = trt.groupby("EOS", observed=True)[LEIDEN].agg(n_clusters="nunique", n_wells="size")
per_cmp = per_cmp[per_cmp.n_wells == 4]
print(f"   compounds with all 4 replicates in one cluster: "
      f"{(per_cmp.n_clusters == 1).mean():.1%} of {len(per_cmp)}")

In [ ]:

print("3. Are annotated tubulin binders concentrated somewhere? (positive control)")
ct = pd.crosstab(obs[LEIDEN], obs.tubulin_binder)
if True in ct.columns:
    ct["enrichment"] = (ct[True] / ct.sum(axis=1)) / obs.tubulin_binder.mean()
    print(ct.sort_values("enrichment", ascending=False).head(6).round(2).to_string())


The 20 annotated tubulin binders are the closest thing to ground truth in this dataset.
If they do not concentrate in one or two clusters, revisit the confound audit before
trusting any of the other clusters. Note that the annotation is noisy — several EGFR
inhibitors (lapatinib, pelitinib, canertinib) carry the tubulin flag.


## Key takeaways

1. **Everything from PCA onward transfers unchanged.** Everything before it does not:
   no `log1p`, no size factors, no `scale`, no HVG.
2. **Clip the heavy tails, or nothing else works.** Unclipped, PC1 is ~93% variance and
   describes a division artifact in `ObjectSkeleton`/`Neighbors` features rather than a
   phenotype.
3. **Well position outranks plate as a confound**, because per-plate normalization
   cannot remove a gradient that recurs at the same coordinates on every plate.
4. **Cell count drives PC1** (r ~ 0.57). Check every cluster against it before calling
   anything a mechanism.
5. **A cluster is a morphological state, not a cell type.** Expect one huge inactive
   cluster; the value is in chemically unrelated compounds sharing a small one.
6. **Quantify before visualising.** Neighbour enrichment tells you how much of the UMAP
   is real: same-compound wells are ~14x enriched, but still only ~8% of neighbours.

## Things to try

- Drop low-DMSO-MAD features instead of clipping, and rerun the confound audit.
- `sc.pp.regress_out` on `cell_count`; compare tubulin-binder enrichment before/after.
- Prune correlated texture features with `pycytominer.feature_select`.
- Cluster *compound consensus profiles* (median over 4 replicates) instead of wells —
  10,668 wells becomes 2,459 compounds and the inactive blob shrinks.
- Bring in a second EU-OS site (`FMP_HepG2`, `MEDINA_HepG2`, `USC_HepG2`) and ask whether
  clusters replicate across sites — the real test of a morphological state.

## References

- [Single-cell best practices](https://www.sc-best-practices.org/)
- Wolff et al. 2024, *EU-OS bioactives Cell Painting* — https://doi.org/10.1101/2024.08.27.609964
- [scverse/cell-painting-io](https://github.com/scverse/cell-painting-io) — the AnnData builder
- Chandrasekaran et al. 2021, *Image-based profiling for drug discovery* — https://doi.org/10.1038/s41573-020-00117-w